# 🎯 AI-Enhanced FPGA Based Smart Surveillance System
## Real-Time Object Detection on PYNQ-Z2

---

| Item | Details |
|---|---|
| **Board** | PYNQ-Z2 FPGA |
| **Camera** | External USB Camera |
| **OS** | PYNQ Linux (Jupyter Notebook) |
| **Detection** | HOG People + Face + Body + Car + Motion |
| **Libraries** | OpenCV, NumPy, Matplotlib |

---

### 📌 How This System Works

- The **PYNQ-Z2 FPGA board** runs a full Linux OS with Python and Jupyter Notebook.
- An **external USB camera** is plugged into one of the USB-A ports on the board.
- OpenCV captures live video frames and runs **5 detection algorithms** in parallel:
  - 🟢 **HOG People Detector** — detects full human bodies
  - 🔵 **Haar Face Detector** — detects faces
  - 🟡 **Haar Upper Body Detector** — detects upper bodies
  - 🟣 **Haar Full Body Detector** — detects full bodies
  - 🔴 **MOG2 Motion Detector** — detects any moving objects
- Results are drawn as **colored bounding boxes** and displayed inside this notebook.


---
## 📦 Cell 1 — Install Required Libraries
Run this cell **once** before anything else.

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1 — Install Required Libraries
# Run this cell once before anything else.
# ══════════════════════════════════════════════════════
import sys

print('Installing just Tesseract OCR...')
!sudo apt-get update -y -q && sudo apt-get install -y -q tesseract-ocr

print('\nInstalling Python wrapper...')
!{sys.executable} -m pip install pytesseract --quiet

print('\n✅ Tesseract installed successfully!')


---
## 📥 Cell 2 — Import Libraries & Verify

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 2 — Import Libraries & Verify
# ══════════════════════════════════════════════════════
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
import time, os, urllib.request
from IPython.display import clear_output

print(f'OpenCV  : {cv2.__version__}')
print(f'NumPy   : {np.__version__}')

# Tesseract OCR check
OCR_AVAILABLE = False
try:
    import pytesseract
    pytesseract.get_tesseract_version()
    OCR_AVAILABLE = True
    print('Tesseract: OK  -> REVA text detection ENABLED')
except Exception as e:
    print(f'Tesseract error details: {e}')
    print('Tesseract: NOT installed -> REVA text detection DISABLED')

print('\n🎯 Imports done!')


---
## ⚙️ Cell 3 — Configuration
Change `CAMERA_INDEX` if your USB camera is not detected at index `0`.

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 3 — Configuration
# ══════════════════════════════════════════════════════

CAMERA_INDEX     = 0        # USB camera index: try 0, 1 or 2
FRAME_WIDTH      = 640      # capture width  (pixels)
FRAME_HEIGHT     = 480      # capture height (pixels)
DISPLAY_FPS      = 15       # notebook render rate (higher = smoother video)
MAX_FRAMES       = 300      # frames before auto-stop  (None = infinite)

# ── YOLO settings ────────────────────────────────────────
YOLO_CONFIDENCE  = 0.45     # min detection confidence (0–1)
YOLO_NMS_THRESH  = 0.40     # Non-Maximum Suppression threshold
YOLO_INPUT_SIZE  = (416, 416)  # YOLO network input size

# ── REVA detection settings ──────────────────────────────
REVA_KEYWORDS    = ['REVA', 'reva']   # text strings to look for
OCR_EVERY_N      = 10       # run OCR every N frames (heavy, so skip frames)

# ── ORB logo matching (set path to your REVA logo image) ─
REVA_LOGO_PATH   = '/home/xilinx/reva_logo.jpg'  # path on PYNQ board

# ── Motion detection ─────────────────────────────────────
MOTION_MIN_AREA  = 1500

# ── YOLO files ───────────────────────────────────────────
YOLO_WEIGHTS     = '/home/xilinx/yolov4-tiny.weights'
YOLO_CFG         = '/home/xilinx/yolov4-tiny.cfg'
COCO_NAMES       = '/home/xilinx/coco.names'

print('✅ Configuration set:')
print(f'   Camera       : index {CAMERA_INDEX}')
print(f'   Resolution   : {FRAME_WIDTH} x {FRAME_HEIGHT}')
print(f'   YOLO conf    : {YOLO_CONFIDENCE}')
print(f'   REVA keywords: {REVA_KEYWORDS}')
print(f'   Max frames   : {MAX_FRAMES}')


---
## 🔍 Cell 4 — Load All Detectors

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 4 — Download YOLO Files & Load All Detectors
# ══════════════════════════════════════════════════════
import os, urllib.request

# ── Download YOLO model files if missing ─────────────────
YOLO_FILES = {
    YOLO_WEIGHTS : 'https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v4_pre/yolov4-tiny.weights',
    YOLO_CFG     : 'https://raw.githubusercontent.com/AlexeyAB/darknet/master/cfg/yolov4-tiny.cfg',
    COCO_NAMES   : 'https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names',
}

for fpath, url in YOLO_FILES.items():
    if not os.path.exists(fpath):
        print(f'Downloading {os.path.basename(fpath)} ...')
        try:
            urllib.request.urlretrieve(url, fpath)
            print(f'  OK -> {fpath}')
        except Exception as e:
            print(f'  ERROR: {e}')
    else:
        print(f'  Already exists: {fpath}')

# ── Load COCO class names ─────────────────────────────────
with open(COCO_NAMES, 'r') as f:
    COCO_CLASSES = [line.strip() for line in f.readlines()]
print(f'\n✅ COCO classes loaded: {len(COCO_CLASSES)} classes')

# ── Auto-assign a unique BGR color per class ──────────────
np.random.seed(42)
COCO_COLORS = {cls: tuple(int(c) for c in np.random.randint(50, 255, 3))
               for cls in COCO_CLASSES}
# Override important ones with vivid colors
COCO_COLORS['person']     = (0, 220, 0)     # Green
COCO_COLORS['car']        = (255, 0, 200)   # Magenta
COCO_COLORS['motorbike']  = (0, 180, 255)   # Orange
COCO_COLORS['bicycle']    = (255, 200, 0)   # Cyan
COCO_COLORS['dog']        = (100, 255, 100) # Light Green
COCO_COLORS['cat']        = (200, 100, 255) # Purple

# ── Load YOLOv4-tiny via OpenCV DNN ──────────────────────
net = cv2.dnn.readNetFromDarknet(YOLO_CFG, YOLO_WEIGHTS)
net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
layer_names = net.getLayerNames()
output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers().flatten()]
print('✅ YOLOv4-tiny loaded      -> Detects ALL 80 COCO objects')

# ── MOG2 background subtractor (motion) ───────────────────
bg_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500, varThreshold=25, detectShadows=True
)
print('✅ MOG2 Motion Detector    -> Detects any moving object (RED)')


# ── ORB: Load REVA logo template (optional) ───────────────
orb_detector = cv2.ORB_create(nfeatures=1000)
logo_kp, logo_des, logo_img = None, None, None
if os.path.exists(REVA_LOGO_PATH):
    logo_img = cv2.imread(REVA_LOGO_PATH, cv2.IMREAD_GRAYSCALE)
    logo_kp, logo_des = orb_detector.detectAndCompute(logo_img, None)
    print(f'✅ REVA Logo Template      -> Loaded from {REVA_LOGO_PATH} ({len(logo_kp)} keypoints)')
else:
    print(f'⚠️  REVA logo template not found at {REVA_LOGO_PATH}')
    print('   -> Place your REVA logo JPG at that path to enable logo matching')

print('\n🎯 All detectors ready!')


---
## 🛠️ Cell 5 — Helper Functions

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 5 — Optimized Helper Functions for Smooth Video
# ══════════════════════════════════════════════════════

def draw_box(frame, x, y, w, h, label, color):
    cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
    lsize, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
    cv2.rectangle(frame, (x, y - lsize[1] - 8), (x + lsize[0] + 4, y), color, -1)
    cv2.putText(frame, label, (x + 2, y - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

# ── HEAVY: YOLO (all 80 classes) ─────────────────────────
def run_yolo(frame):
    H, W = frame.shape[:2]
    # Resize to 320 for speed (standard YOLO is 416 but PYNQ needs speed)
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (320, 320), swapRB=True, crop=False)
    net.setInput(blob)
    outs = net.forward(output_layers)
    boxes, confs, cids = [], [], []
    for out in outs:
        for det in out:
            scores = det[5:]
            cid, conf = np.argmax(scores), float(scores[np.argmax(scores)])
            if conf > YOLO_CONFIDENCE:
                cx, cy, bw, bh = (det[:4] * [W, H, W, H]).astype(int)
                boxes.append([max(0, cx - bw//2), max(0, cy - bh//2), int(bw), int(bh)])
                confs.append(conf)
                cids.append(cid)
    indices = cv2.dnn.NMSBoxes(boxes, confs, YOLO_CONFIDENCE, YOLO_NMS_THRESH)
    return [(COCO_CLASSES[cids[i]], confs[i], *boxes[i]) for i in (indices.flatten() if len(indices)>0 else [])]

# ── FAST: Motion (MOG2) ──────────────────────────────────
def run_motion(frame):
    fg = bg_subtractor.apply(frame)
    _, fg = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)
    cnts, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return [cv2.boundingRect(c) for c in cnts if cv2.contourArea(c) > MOTION_MIN_AREA]

# ── HEAVY: REVA Text (Tesseract) ─────────────────────────
def run_reva_text(frame):
    if not OCR_AVAILABLE: return []
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    try:
        data = pytesseract.image_to_data(gray, output_type=pytesseract.Output.DICT)
        return [(data['left'][i], data['top'][i], data['width'][i], data['height'][i], data['text'][i]) 
                for i in range(len(data['text'])) 
                if int(data['conf'][i]) > 30 and any(kw.lower() in data['text'][i].lower() for kw in REVA_KEYWORDS)]
    except: return []

# ── MEDIUM: REVA Logo (ORB) ──────────────────────────────
def run_reva_logo(frame):
    if logo_des is None: return None
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    try:
        kp2, des2 = orb_detector.detectAndCompute(gray, None)
        if des2 is None or len(des2) < 5: return None
        matches = sorted(cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True).match(logo_des, des2), key=lambda x:x.distance)
        good = [m for m in matches if m.distance < 60]
        if len(good) >= 8:
            pts = np.float32([kp2[m.trainIdx].pt for m in good])
            x, y = int(pts[:,0].min()), int(pts[:,1].min())
            return (x, y, max(60, int(pts[:,0].max())-x), max(60, int(pts[:,1].max())-y))
    except: return None

# ── FAST DISPLAY ─────────────────────────────────────────
def show_in_notebook(frame, frame_count, fps, dhandle=None):
    _, buffer = cv2.imencode('.jpg', frame, [int(cv2.IMWRITE_JPEG_QUALITY), 75])
    img_data = buffer.tobytes()
    if dhandle is None:
        dhandle = display(Image(data=img_data), display_id=True)
    else:
        dhandle.update(Image(data=img_data))
    return dhandle

print('✅ Optimized helpers ready!')


---
## ▶️ Cell 6 — Run the Surveillance System

**What you will see:**
- 🟢 **Green box** → Person (HOG detected)
- 🔵 **Blue box** → Face
- 🟡 **Cyan box** → Upper Body
- 🟨 **Yellow box** → Full Body
- 🟣 **Magenta box** → Car
- 🔴 **Red box** → Motion Region

> ⏹️ **To stop early**: Use `Kernel → Interrupt` in the Jupyter menu bar.

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 6 — High-Performance Live Video Loop
# ══════════════════════════════════════════════════════

def run_surveillance(max_frames=MAX_FRAMES):
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened(): return print('ERROR: Camera failed')
    
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
    
    # --- Persistent Detection State ---
    cache = {'yolo': [], 'text': [], 'logo': None, 'summary': {}}
    
    frame_count = 0
    prev_time = time.time()
    dh = None
    
    print('🚀 LIVE FEED STARTED! Decoupled AI mode.')
    print('   - Video: Smooth Live Feed')
    print('   - AI   : Periodic Background Check')

    try:
        while True:
            ret, frame = cap.read()
            if not ret: continue
            frame_count += 1
            
            # 1. RUN HEAVY AI periodically (every 50 frames)
            # or on first frame
            if frame_count % 50 == 1 or frame_count == 1:
                cache['yolo'] = run_yolo(frame)
                cache['text'] = run_reva_text(frame)
                cache['logo'] = run_reva_logo(frame)
                
                # Update summary
                s = {'YOLO Objects': len(cache['yolo']), 'REVA Text': len(cache['text']), 'REVA Logo': 1 if cache['logo'] else 0}
                counts = {}
                for (cls, *rest) in cache['yolo']: counts[cls] = counts.get(cls, 0) + 1
                s['Top'] = ', '.join(f'{k}:{v}' for k, v in sorted(counts.items(), key=lambda x:-x[1])[:2])
                cache['summary'] = s
                if s['REVA Text'] or s['REVA Logo']: print(f'*** REVA SPOTTED @ frame {frame_count} ***')

            # 2. RUN FAST AI every frame
            motion = run_motion(frame)
            
            # 3. DRAW COMBINED RESULTS
            annotated = frame.copy()
            # Motion (Live)
            for x, y, w, h in motion: draw_box(annotated, x, y, w, h, 'Motion', (0,0,225))
            # YOLO (Cached)
            for cls, conf, x, y, w, h in cache['yolo']: draw_box(annotated, x, y, w, h, f'{cls}', COCO_COLORS.get(cls, (255,255,255)))
            # REVA Text (Cached)
            for x, y, w, h, txt in cache['text']: draw_box(annotated, x, y, w, h, f'REVA: {txt}', (0,255,255))
            # REVA Logo (Cached)
            if cache['logo']: x, y, w, h = cache['logo']; draw_box(annotated, x, y, w, h, 'REVA LOGO', (255,125,0))

            # 4. HUD & DISPLAY
            curr_time = time.time()
            fps = 1.0 / max(curr_time - prev_time, 1e-6)
            prev_time = curr_time
            
            # Title/FPS Overlay
            cv2.putText(annotated, f'LIVE | {fps:.1f} FPS | Frame #{frame_count}', (10, 25), 1, 1.2, (0,255,0), 2)
            cv2.putText(annotated, f'AI Mode: Decoupled | Next AI in {50 - (frame_count%50)} fr', (10, 50), 1, 1.0, (200,200,200), 1)
            
            # Show in notebook every ~2 frames to avoid bandwidth lag
            if frame_count % 2 == 0:
                dh = show_in_notebook(annotated, frame_count, fps, dh)

            if max_frames and frame_count >= max_frames: break
            
    except KeyboardInterrupt: pass
    finally: cap.release(); print('Done.')

run_surveillance()


---
## 💾 Cell 7 — Optional: Save Detected Frames to Disk

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 7 — Optional: Save Detected Frames to Disk
# Saves a .jpg only when REVA or any YOLO object is detected
# ══════════════════════════════════════════════════════
import os

SAVE_DIR   = '/home/xilinx/surveillance_captures'
SAVE_LIMIT = 100
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Saving detected frames to: {SAVE_DIR}')

cap2 = cv2.VideoCapture(CAMERA_INDEX)
cap2.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
cap2.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

saved, fc = 0, 0
try:
    while saved < SAVE_LIMIT:
        ret, frame = cap2.read()
        if not ret:
            break
        fc += 1
        annotated, summary = detect_all_objects(frame, fc)
        total = summary.get('YOLO Objects', 0) + summary.get('REVA Text', 0) + summary.get('REVA Logo', 0)
        if total > 0:
            fname = os.path.join(SAVE_DIR, f'detection_{saved:04d}.jpg')
            cv2.imwrite(fname, annotated)
            saved += 1
            print(f'  Saved: {fname} | {summary}')
except KeyboardInterrupt:
    print('Stopped by user.')
finally:
    cap2.release()
    print(f'Done! {saved} images saved to {SAVE_DIR}')


---
## 📖 Cell 8 — System Reference

### 🔌 Hardware
| Component | Connection |
|---|---|
| USB Camera | PYNQ-Z2 USB-A port |
| Laptop | PYNQ-Z2 Ethernet (IP: 192.168.2.99) |
| Power | DC barrel 12V or micro-USB 5V |

### 🔍 What This System Detects
| Detector | Objects Detected | Box Color |
|---|---|---|
| YOLOv4-tiny | ALL 80 COCO objects (person, car, dog, bottle, phone…) | Per-class unique color |
| EasyOCR | **REVA text on cards / t-shirts** | 🟡 Bright Yellow |
| ORB Matching | **REVA logo on cards / t-shirts** | 🟠 Orange |
| MOG2 Motion | Any moving object | 🔴 Red |

### 📋 80 COCO Object Classes
person, bicycle, car, motorbike, aeroplane, bus, train, truck, boat,
traffic light, fire hydrant, stop sign, parking meter, bench, bird,
cat, dog, horse, sheep, cow, elephant, bear, zebra, giraffe, backpack,
umbrella, handbag, tie, suitcase, frisbee, skis, snowboard, sports ball,
kite, baseball bat, baseball glove, skateboard, surfboard, tennis racket,
bottle, wine glass, cup, fork, knife, spoon, bowl, banana, apple,
sandwich, orange, broccoli, carrot, hot dog, pizza, donut, cake, chair,
sofa, pottedplant, bed, diningtable, toilet, tvmonitor, laptop, mouse,
remote, keyboard, cell phone, microwave, oven, toaster, sink,
refrigerator, book, clock, vase, scissors, teddy bear, hair drier, toothbrush

### ⚙️ REVA Detection Tips
| Scenario | What To Do |
|---|---|
| REVA text on t-shirt not detected | Improve lighting, hold card still, wait for OCR frame |
| REVA logo not detected | Place your REVA logo JPG at `/home/xilinx/reva_logo.jpg` |
| Too slow on PYNQ-Z2 | Increase `OCR_EVERY_N` to 20 or 30 |
| YOLO detects wrong objects | Increase `YOLO_CONFIDENCE` to 0.55+ |
| Too few detections | Decrease `YOLO_CONFIDENCE` to 0.35 |
